# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima12aa/fa-ml/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token
print("Token loaded:", hf_token[:8] + "..." if hf_token else "NOT FOUND")

Token loaded: hf_DCVCv...


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Tables defined.")

Tables defined.


In [ ]:
feb_ctr = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(gsc_impressions) AS feb_impressions,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
            ELSE NULL
        END AS feb_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_position = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS feb_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
      AND gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

print("feb_ctr:", feb_ctr.shape)
print("feb_position:", feb_position.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feb_ctr: (153559, 4)
feb_position: (151956, 2)


In [ ]:
feb_position["feb_avg_position"].describe()

,feb_avg_position
count,151956.000000
mean,14.133842
std,15.256815
min,0.057700
25%,5.348455
50%,8.647976
75%,16.500000
max,633.000000


feb_ctr table (153,559 rows, 4 columns): content_hash_id (the page identifier), feb_clicks, feb_impressions, feb_ctr (the computed ratio).
feb_position table (151,956 rows, 2 columns): content_hash_id, feb_avg_position.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Token loaded: hf_DCVCv...


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.